# Travel AI — Demo Backend

Ce notebook permet de tester **chaque brique du backend** de Travel AI, étape par étape.

### Ce qu'on va tester :
1. **Carbon Service** — Calcul des émissions CO₂ train vs avion (aucune clé API nécessaire)
2. **Affiliate Service** — Génération de liens de réservation (aucune clé API nécessaire)
3. **Duffel API** — Recherche de vols réels (nécessite `DUFFEL_ACCESS_TOKEN`)
4. **Gemini AI** — Le cerveau : parsing d'intent + réponses (nécessite `GEMINI_API_KEY`)
5. **Flow complet** — Un utilisateur envoie un message → Gemini comprend → appelle les outils → répond

---

### Pré-requis
- Le fichier `.env` à la racine du projet doit contenir les clés API
- Le venv `travel-ai` doit être installé (c'est fait si tu vois ce notebook)

In [ ]:
# Setup : charger les variables d'environnement et ajouter le projet au path
import sys, os
from dotenv import load_dotenv

# Ajouter le dossier racine au path Python pour que "backend" soit importable
project_root = os.path.dirname(os.path.abspath("__file__"))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

load_dotenv()
print("✅ Environnement chargé")
print(f"   GEMINI_API_KEY : {'✅ présente' if os.getenv('GEMINI_API_KEY') else '❌ manquante'}")
print(f"   DUFFEL_ACCESS_TOKEN : {'✅ présent' if os.getenv('DUFFEL_ACCESS_TOKEN') else '❌ manquant'}")

---

## 1. Carbon Service — Comparaison CO₂ train vs avion

Ce service calcule les émissions carbone pour n'importe quel trajet entre villes européennes.  
Il utilise les données officielles **ADEME 2023** (Agence française de la transition écologique).

**Aucune clé API nécessaire** — tout est calculé localement.

In [ ]:
from backend.services.carbon_service import get_distance_km, calculate_co2, co2_equivalence, compare_carbon

# Testons quelques routes populaires
routes = [
    ("Paris", "Barcelona"),
    ("Paris", "Rome"),
    ("London", "Amsterdam"),
    ("Berlin", "Prague"),
    ("Paris", "London"),
]

print("🌍 Distances entre villes européennes")
print("-" * 40)
for origin, dest in routes:
    dist = get_distance_km(origin, dest)
    print(f"  {origin:10} → {dest:12} : {dist:,.0f} km")

In [ ]:
# Comparaison CO₂ détaillée : Paris → Barcelona
result = compare_carbon("Paris", "Barcelona")

print("🚄 vs ✈️  — Paris → Barcelona")
print("=" * 50)
print(f"  Distance         : {result['distance_km']} km")
print()
print(f"  🚄 Train (TGV)   : {result['train_co2_grams']/1000:.1f} kg CO₂")
print(f"  ✈️  Avion         : {result['flight_co2_grams']/1000:.1f} kg CO₂")
print()
print(f"  💚 Économie train : -{result['savings_pct']}% ({result['savings_grams']/1000:.1f} kg)")
print()
print(f"  En prenant l'avion, tu émets autant que :")
print(f"    {co2_equivalence(result['flight_co2_grams'])}")
print(f"  En prenant le train :")
print(f"    {co2_equivalence(result['train_co2_grams'])}")

In [ ]:
# Comparaison pour toutes les routes
print("📊 Tableau comparatif CO₂ — Principales routes européennes")
print("=" * 75)
print(f"  {'Route':25} {'Distance':>8}  {'Train':>10}  {'Avion':>10}  {'Économie':>10}")
print("-" * 75)

for origin, dest in routes:
    r = compare_carbon(origin, dest)
    if r:
        print(
            f"  {origin+'→'+dest:25} {r['distance_km']:>7.0f}km"
            f"  {r['train_co2_grams']/1000:>8.1f}kg"
            f"  {r['flight_co2_grams']/1000:>8.1f}kg"
            f"  {'-'+str(r['savings_pct'])+'%':>9}"
        )

---

## 2. Affiliate Service — Liens de réservation

Génère des URLs de réservation vers les partenaires (Trainline, Booking.com, GetYourGuide).  
En production, ces liens contiendront un ID de tracking affilié pour les commissions.

**Aucune clé API nécessaire.**

In [ ]:
from backend.services.affiliate import trainline_url, booking_com_url, get_your_guide_url

# Exemple : voyage à Barcelona du 1er au 5 mai
print("🔗 Liens de réservation générés pour un voyage Paris → Barcelona")
print("=" * 70)
print()
print("🚄 Réserver le train (Trainline) :")
print(f"   {trainline_url('Paris', 'Barcelona', '2026-05-01')}")
print()
print("🏨 Réserver l'hôtel (Booking.com) :")
print(f"   {booking_com_url('Barcelona', '2026-05-01', '2026-05-05')}")
print()
print("🎭 Réserver des activités (GetYourGuide) :")
print(f"   {get_your_guide_url('Barcelona')}")

---

## 3. Duffel API — Recherche de vols réels

Duffel connecte 300+ compagnies aériennes et retourne des **prix réels en temps réel**.  
Le SDK Python officiel étant abandonné (archivé sept. 2024), on utilise l'API REST directement via `httpx`.

**Nécessite `DUFFEL_ACCESS_TOKEN`** dans le `.env` (mode test = données réelles sans paiement).

In [ ]:
from datetime import date, timedelta
from backend.services.duffel_service import DuffelService
from backend.services.carbon_service import calculate_co2, get_distance_km

duffel = DuffelService()

# Recherche de vols : Paris (CDG) → Barcelona (BCN), dans 2 semaines
dep_date = date.today() + timedelta(days=100)
print(f"✈️  Recherche de vols : Paris CDG → Barcelona BCN")
print(f"   Date : {dep_date}")
print(f"   Passagers : 1 adulte")
print()

flights = await duffel.search_flights(
    origin="CDG",
    destination="BCN",
    departure_date=dep_date,
    passengers=1,
)

# Enrichir avec les données carbone
distance = get_distance_km("Paris", "Barcelona")
for flight in flights:
    flight.co2_grams = calculate_co2("flight", distance)

print(f"📋 {len(flights)} offres trouvées. Top 10 :")
print("=" * 80)
print(f"  {'Compagnie':20} {'Départ':>7} {'Arrivée':>7} {'Durée':>7} {'Prix':>8} {'CO₂':>8} {'Escales':>7}")
print("-" * 80)

for f in flights[:10]:
    price = f"€{f.price_eur:.0f}" if f.price_eur else "N/A"
    print(
        f"  {f.provider:20}"
        f" {f.departure.strftime('%H:%M'):>7}"
        f" {f.arrival.strftime('%H:%M'):>7}"
        f" {f.duration_minutes//60}h{f.duration_minutes%60:02d}m"
        f" {price:>8}"
        f" {f.co2_grams/1000:.0f}kg"
        f" {f.transfers:>7}"
    )

In [ ]:
# Comparaison avec le train pour la même route
train_co2 = calculate_co2("train", distance)
flight_co2 = flights[0].co2_grams if flights else 0
cheapest = flights[0].price_eur if flights else 0

print("🚄 vs ✈️  — Ce que Travel AI montrera à l'utilisateur")
print("=" * 60)
print()
print(f"  ✈️  Vol le moins cher : €{cheapest:.0f}")
print(f"     CO₂ : {flight_co2/1000:.0f} kg par passager")
print()
print(f"  🚄 Train (TGV direct, ~6h30) : ~€39-180 (estimé)")
print(f"     CO₂ : {train_co2/1000:.1f} kg par passager")
print()
print(f"  💚 En prenant le train, tu économises {(flight_co2-train_co2)/1000:.0f} kg de CO₂")
print(f"     C'est -{((flight_co2-train_co2)/flight_co2*100):.0f}% d'émissions !")

---

## 4. Gemini AI — Le cerveau de Travel AI

Gemini (Google) est le LLM qui alimente Travel AI. Il a deux super-pouvoirs :

### 4a. Comprendre ce que l'utilisateur veut (intent parsing)
L'utilisateur dit en langage naturel "Je veux aller à Barcelone pas cher début mai", et Gemini transforme ça en données structurées (ville départ, ville arrivée, dates, budget...) via le **function calling**.

### 4b. Répondre de manière naturelle
Après avoir récupéré les données (vols, CO₂, etc.), Gemini rédige une réponse en français, enthousiaste mais pas moralisatrice.

**Nécessite `GEMINI_API_KEY`** dans le `.env`.

In [ ]:
from backend.services.gemini_service import GeminiService
import json

gemini = GeminiService()

# Test 1 : Message simple — Gemini doit répondre directement, sans tool call
print("💬 Test 1 : Question simple (pas de tool call attendu)")
print("=" * 60)
print('   Message : "Salut ! C\'est quoi Travel AI ?"')
print()

response = await gemini.chat("Salut ! C'est quoi Travel AI ?")

print("   Réponse de Gemini :")
print()
for line in (response.text or "").split("\n"):
    print(f"   {line}")

In [ ]:
# Test 2 : Intent parsing — Gemini doit appeler parse_travel_intent
print("🧠 Test 2 : Parsing d'intent (function calling)")
print("=" * 60)

message = "Je veux aller de Paris à Barcelone le 10 mai, 5 jours, budget serré"
print(f'   Message : "{message}"')
print()

response = await gemini.chat(message)

# Chercher les tool calls dans la réponse
tool_calls = []
if response.candidates:
    for part in response.candidates[0].content.parts:
        if part.function_call:
            tool_calls.append({
                "name": part.function_call.name,
                "args": dict(part.function_call.args) if part.function_call.args else {},
            })

if tool_calls:
    print(f"   ✅ Gemini a appelé {len(tool_calls)} outil(s) :")
    print()
    for tc in tool_calls:
        print(f"   🔧 {tc['name']}")
        print(f"   Arguments :")
        for key, val in tc['args'].items():
            print(f"      {key}: {val}")
        print()
    
        # Si c'est parse_travel_intent, montrer le TravelIntent résultant
        if tc['name'] == 'parse_travel_intent':
            intent = gemini.parse_intent_from_tool_call(tc['args'])
            print("   📋 TravelIntent résultant :")
            print(f"      Départ      : {intent.origin}")
            print(f"      Destination  : {intent.destination}")
            print(f"      Date départ  : {intent.departure_date}")
            print(f"      Durée        : {intent.duration_days} jours")
            print(f"      Budget       : {intent.budget}")
            print(f"      Flexibilité  : {intent.flexibility}")
            print(f"      Préférences  : {intent.preferences}")
else:
    print("   ℹ️  Gemini a répondu directement (pas de tool call) :")
    print(f"   {(response.text or '')[:300]}")

---

## 5. Flow complet — La magie de Travel AI

Voici le flux complet tel qu'il fonctionnera dans l'application :

1. L'utilisateur tape un message en langage naturel
2. Gemini comprend l'intent et appelle `parse_travel_intent`
3. On exécute les outils (recherche de transports, calcul CO₂)
4. On renvoie les résultats à Gemini
5. Gemini rédige une réponse finale naturelle

C'est exactement ce qui se passe dans l'endpoint `POST /api/chat`.

In [ ]:
from backend.services.gemini_service import GeminiService
from backend.services.duffel_service import DuffelService
from backend.routers.chat import _handle_tool_calls

gemini = GeminiService()
duffel = DuffelService()

# Simule un utilisateur qui envoie un message
message = "Je veux aller de Paris à Rome la deuxième semaine de juillet, 4 jours"

print("👤 UTILISATEUR :")
print(f'   "{message}"')
print()
print("⏳ Traitement en cours...")
print("   [1/3] Envoi à Gemini...")

# Étape 1 : Gemini analyse le message
response = await gemini.chat(message)

# Étape 2 : On exécute les tool calls et on renvoie les résultats
print("   [2/3] Exécution des outils...")
text, intent, tool_calls = await _handle_tool_calls(message, response, gemini, duffel)

print("   [3/3] Réponse générée !")
print()

# Affichage des résultats
if intent:
    print("📋 INTENT DÉTECTÉ :")
    print(f"   {intent.origin} → {intent.destination}")
    print(f"   Durée : {intent.duration_days} jours")
    if intent.budget:
        print(f"   Budget : {intent.budget}")
    print()

if tool_calls:
    print(f"🔧 OUTILS APPELÉS : {[tc['tool'] for tc in tool_calls]}")
    for tc in tool_calls:
        if tc['tool'] == 'get_carbon_comparison' and tc.get('result'):
            r = tc['result']
            print(f"   Carbone : train {r.get('train_co2_grams', 0)/1000:.1f}kg vs avion {r.get('flight_co2_grams', 0)/1000:.1f}kg")
    print()

print("🤖 RÉPONSE DE TRAVEL AI :")
print("-" * 60)
print(text)

---

## 6. Essaie toi-même !

Change le message ci-dessous et relance la cellule pour voir comment Travel AI réagit à différentes requêtes.

In [ ]:
# ✏️ MODIFIE CE MESSAGE et relance cette cellule (Shift+Enter)
mon_message = "Je voudrais partir de Lyon vers Amsterdam, 3 jours, budget modéré, en train si possible"

# --- ne touche pas en dessous ---
from backend.services.gemini_service import GeminiService
from backend.services.duffel_service import DuffelService
from backend.routers.chat import _handle_tool_calls

gemini = GeminiService()
duffel = DuffelService()

print(f'👤 "{mon_message}"')
print()

response = await gemini.chat(mon_message)
text, intent, tool_calls = await _handle_tool_calls(mon_message, response, gemini, duffel)

if intent:
    print(f"📋 Compris : {intent.origin} → {intent.destination}, {intent.duration_days}j")
    if intent.budget:
        print(f"   Budget : {intent.budget}")
    if intent.preferences:
        print(f"   Préférences : {', '.join(intent.preferences)}")
    print()

print("🤖 Réponse :")
print("-" * 60)
print(text)